In [1]:
import torch
import numpy as np

In [2]:
# Synthetic Data
num_samples_per_class = 1000
negative_samples = np.random.multivariate_normal(mean=[0, 3], cov=[[1, 0.55], [0.55, 1]], size=num_samples_per_class)
positive_samples = np.random.multivariate_normal(mean=[3, 0], cov=[[1, 0.55], [0.55, 1]], size=num_samples_per_class)
inputs = np.vstack((negative_samples, positive_samples)).astype(np.float32)
targets = np.vstack((np.zeros((num_samples_per_class, 1), dtype="float32"), np.ones((num_samples_per_class, 1), dtype="float32")))


In [3]:
input_dim = 2
output_dim = 1
W = torch.rand(input_dim, output_dim, requires_grad=True)
b = torch.zeros(output_dim, requires_grad=True)

In [4]:
W

tensor([[0.6413],
        [0.6529]], requires_grad=True)

In [5]:
b

tensor([0.], requires_grad=True)

In [6]:
def model(inputs, W, b):
    return torch.matmul(inputs, W) + b

In [7]:
def mean_squared_error(targets, predictions):
    per_sample_losses = torch.square(targets - predictions)
    return torch.mean(per_sample_losses)

In [8]:
learning_rate = 0.1
def training_step(inputs, targets, W, b):
    predictions = model(inputs, W, b)
    loss = mean_squared_error(targets, predictions)
    loss.backward()
    grad_loss_wrt_W, grad_loss_wrt_b = W.grad, b.grad
    with torch.no_grad():
        W -= grad_loss_wrt_W * learning_rate
        b -= grad_loss_wrt_b * learning_rate
    W.grad = None
    b.grad = None
    return loss

In [9]:
epoches = 50
inputs = torch.tensor(inputs)
targets = torch.tensor(targets)
for epoch in range(epoches):
    loss = training_step(inputs, targets, W, b)
    if epoch%10 ==0:
        print(f"Loss at step {epoch}: {loss:.4f}")


Loss at step 0: 3.6736
Loss at step 10: 0.0779
Loss at step 20: 0.0448
Loss at step 30: 0.0318
Loss at step 40: 0.0268


In [10]:
predictions = model(inputs, W, b)
predicted_labels = predictions[:, 0] > 0.5
matches = (predicted_labels == targets.flatten()).to(float)
accuracy = torch.mean(matches).item()
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 99.90%
